In [0]:
CREATE EXTENSION IF NOT EXISTS vector;

In [0]:
CREATE TABLE IF NOT EXISTS weather_documents (
    id VARCHAR(255) PRIMARY KEY,
    location VARCHAR(500) NOT NULL,
    source_type VARCHAR(50) NOT NULL,
    headline TEXT,
    narrative_text TEXT,
    issued_at TIMESTAMP,
    effective_at TIMESTAMP,
    payload JSONB,
    synced_at TIMESTAMP NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

In [0]:
CREATE INDEX IF NOT EXISTS idx_weather_location
ON weather_documents(location);

CREATE INDEX IF NOT EXISTS idx_weather_source_type
ON weather_documents(source_type);

CREATE INDEX IF NOT EXISTS idx_weather_synced_at
ON weather_documents(synced_at);

In [0]:
CREATE TABLE IF NOT EXISTS weather_embeddings (
    id SERIAL PRIMARY KEY,
    document_id VARCHAR(255) NOT NULL
        REFERENCES weather_documents(id)
        ON DELETE CASCADE,
    chunk_index INTEGER NOT NULL,
    chunk_text TEXT NOT NULL,
    embedding VECTOR(384) NOT NULL,
    model_name VARCHAR(255) NOT NULL
        DEFAULT 'sentence-transformers/all-MiniLM-L6-v2',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    UNIQUE(document_id, chunk_index)
);

In [0]:
CREATE INDEX IF NOT EXISTS weather_embeddings_hnsw_idx
ON weather_embeddings
USING hnsw (embedding vector_cosine_ops)
WITH (
    m = 16,
    ef_construction = 64
);

In [0]:
CREATE INDEX IF NOT EXISTS idx_weather_embeddings_document_id
ON weather_embeddings(document_id);

CREATE INDEX IF NOT EXISTS idx_weather_embeddings_created_at
ON weather_embeddings(created_at);

In [0]:
SELECT
    table_name,
    column_name,
    data_type,
    udt_name
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN (
      'weather_documents',
      'weather_embeddings'
  )
ORDER BY table_name, ordinal_position;

In [0]:
SELECT
    vector_dims(embedding) AS embedding_dimension,
    COUNT(*) AS embedding_count
FROM weather_embeddings
GROUP BY vector_dims(embedding);